In [1]:
import wbgapi as wb
import pandas as pd
import numpy as np
from IPython.terminal.shortcuts.filters import pass_through

In [6]:
Indicators= {
    "EG.FEC.RNEW.ZS": "renewable_energy_pct",
    "NY.GDP.PCAP.KD": "gdp_per_capita_usd"
}
countries= ["USA","DEU","BRA","IND","ETH"]
year=range(2010,2017)
print("fetching data from world bak")

fetching data from world bak


the next step is just fetching the data from the world bank data api directly. so no need to download first

In [7]:
raw_df=wb.data.DataFrame(
    list(Indicators.keys()),
    economy=countries,
    time= year,
    labels= True
).reset_index()



convert the wide format into long format( which is very easy for tableau

In [8]:
year_cols = [c for c in raw_df.columns if c.startswith("YR")]
melted_df = pd.melt(
    raw_df,
    id_vars=["economy", "Country", "series", "Series"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
)

melted_df["Year"] = melted_df["Year"].str.replace("YR", "").astype(int)
melted_df["metric_name"] = melted_df["series"].map(Indicators)

# 3. Pivot so each indicator has its own column (standard for Tableau)
tidy_df = melted_df.pivot(
    index=["economy", "Country", "Year"],
    columns="metric_name",
    values="Value"
).reset_index()

check missing value next, as we can see below there is no missing value.

In [9]:
print("missing values")
print(tidy_df.isnull().sum())

missing values
metric_name
economy                 0
Country                 0
Year                    0
gdp_per_capita_usd      0
renewable_energy_pct    0
dtype: int64


In [12]:
output_file= "World_Bank_Excercise.csv"
tidy_df.to_csv(output_file)
